# Regresión Espacial (Parte 2)

## La idea de fondo

Todos los modelos de regresión espacial parten del OLS clásico y le agregan **un término nuevo** que mete el espacio en la ecuación. La pregunta clave es siempre la misma: **¿dónde aparece el $W$?** (la matriz de vecinos). Cuatro opciones, una por modelo:

| Modelo | Ecuación | Término nuevo |
|---|---|---|
| **OLS** (clase 08) | $y = X\beta + \varepsilon$ | (ninguno) |
| **SLX** (clase 08) | $y = X\beta + \color{red}{WX\,\gamma} + \varepsilon$ | $\color{red}{WX\gamma}$ |
| **SEM** (hoy) | $y = X\beta + u, \quad u = \color{red}{\lambda Wu} + \varepsilon$ | $\color{red}{\lambda Wu}$ |
| **SAR** (hoy) | $y = \color{red}{\rho Wy} + X\beta + \varepsilon$ | $\color{red}{\rho Wy}$ |

### Lo mismo en palabras, traducido a Airbnb San Diego

| Modelo | Qué dice, en lenguaje vivencial (manteniendo todo lo demás fijo) |
|---|---|
| **OLS** | El precio depende **solo de los atributos de la propia propiedad** (dormitorios, baños, distancia a Balboa). El barrio no entra al modelo. |
| **SLX** | Además de los atributos propios, **los atributos de las propiedades vecinas importan**: estar rodeado de departamentos grandes encarece el mío, aunque el mío sea chico. |
| **SEM** | Hay características del entorno que **no medimos** (vista al mar, ruido, seguridad, calidad del barrio) y que están agrupadas en el espacio. Su efecto cae en el error, y por eso los errores de propiedades vecinas se parecen. |
| **SAR** | El precio de mi propiedad **depende del precio de las vecinas**: si una zona se encarece, ese efecto se propaga a las propiedades de al lado a través de la red espacial. |

> Hay una quinta forma de incorporar el espacio (dejar que $\beta$ cambie por zona: **regímenes / heterogeneidad espacial**) que veremos en la clase 10.

**En qué fijarse al leer un output de cada modelo:**

- **SLX** → el coeficiente $\gamma_k$ de cada `w_x_k`. Si es significativo, el atributo de los vecinos importa (*spillover exógeno*).
- **SEM** → el parámetro $\lambda$. Si $\lambda \neq 0$, hay variables omitidas territoriales (los $\beta$ no cambian respecto al OLS, **lo que cambia son los p-valores**, que ahora son válidos).
- **SAR** → el parámetro $\rho$. Si $\rho \neq 0$, hay contagio en $y$. Y los $\beta$ **ya no son** los efectos marginales: hay que descomponerlos en **efectos directos e indirectos** usando $(I-\rho W)^{-1}$.

### Qué función de `spreg` se usa para cada uno

| Modelo | Llamado | Atributos clave del resultado |
|---|---|---|
| **OLS** | `spreg.OLS(y, X, w=w)` | `.r2`, `.betas`, `.u` (residuos) |
| **OLS + tests LM** | `spreg.OLS(y, X, w=w, spat_diag=True)` | + `.lm_error`, `.lm_lag`, `.rlm_error`, `.rlm_lag` |
| **SLX** | `spreg.OLS(y, X_con_lags, w=w)` con `X_con_lags = [X, WX]` | igual que OLS, los γ aparecen en `.betas` |
| **SEM** | `spreg.GM_Error_Het(y, X, w=w)` | $\hat\lambda$ en `.betas[-1]`, residuos filtrados en `.e_filtered` |
| **SAR** | `spreg.GM_Lag(y, X, w=w)` | $\hat\rho$ en `.betas[-1]`, pseudo-$R^2$ en `.pr2` |
| **Efectos SAR** | `spreg.spmultiplier(w, rho, method="full")` | dict con `ADI`, `AII`, `ATI` |

## Programa de hoy

1. **Tests LM**: cómo decidir entre SEM y SAR antes de ajustarlos.
2. **SEM**: estimación con `spreg.GM_Error_Het`, interpretación de $\lambda$.
3. **SAR**: estimación con `spreg.GM_Lag`, interpretación de $\rho$, **efectos directos vs indirectos**.
4. **Tabla comparativa** final + regla práctica para elegir modelo.

Seguimos con el mismo dataset de la clase 08 (Airbnb San Diego, $n = 6{,}110$, variable dependiente `log_price`) para que las comparaciones sean limpias.

Adaptado de Rey, Arribas-Bel & Wolf (2020), *Geographic Data Science with Python*, Capítulo 11.

## 1. Setup y recap de la clase 08

Cargamos las mismas librerías, datos y matriz de pesos $W$ (KNN con $k=20$, row-standardized) que en la clase 08. Re-ajustamos OLS y SLX rápidamente para tenerlos como referencia.

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import seaborn as sns
import matplotlib.pyplot as plt
import contextily as cx

from libpysal import weights
from pysal.model import spreg
from esda.moran import Moran

%config InlineBackend.figure_format = "retina"
np.random.seed(42)

sns.set(style='ticks', font='sans-serif', context='notebook', palette='viridis')
plt.rcParams["figure.dpi"] = 96
plt.rcParams["font.family"] = "Fira Sans Extra Condensed"
pd.set_option('display.max_columns', None)

In [2]:
db = gpd.read_file("datos/external/airbnb_sd/regression_db.geojson")

variables = [
    "accommodates", "bathrooms", "bedrooms", "beds", "d2balboa",
    "rt_Private_room", "rt_Shared_room",
    "pg_Condominium", "pg_House", "pg_Other", "pg_Townhouse",
]
y = db[["log_price"]].values
X = db[variables].values

coords = np.column_stack([db.geometry.x, db.geometry.y])
w = weights.KNN.from_array(coords, k=20)
w.transform = "r"

print(f"Propiedades: {len(db)}   Variables: {len(variables)}   Vecinos por punto: {w.mean_neighbors:.0f}")

Propiedades: 6110   Variables: 11   Vecinos por punto: 20


/Users/daniela/Documents/usm/cursos/geodata/.venv/lib/python3.11/site-packages/libpysal/weights/distance.py:153: UserWarning: The weights matrix is not fully connected: 
 There are 3 disconnected components.
  W.__init__(self, neighbors, id_order=ids, **kwargs)


**OLS y SLX (recap)**: ajustamos los dos modelos de la clase 08 y guardamos sus residuos para comparar con los modelos de hoy.

In [3]:
m_ols = spreg.OLS(y, X, w=w, name_y="log_price", name_x=variables)

# SLX: agregamos lag espacial de tres variables
lag_vars = ["accommodates", "bedrooms", "d2balboa"]
for v in lag_vars:
    db[f"w_{v}"] = weights.lag_spatial(w, db[v].values)

variables_slx = variables + [f"w_{v}" for v in lag_vars]
X_slx = db[variables_slx].values
m_slx = spreg.OLS(y, X_slx, w=w, name_y="log_price", name_x=variables_slx)

# Moran de residuos
mi_ols = Moran(m_ols.u.flatten(), w)
mi_slx = Moran(m_slx.u.flatten(), w)
print(f"OLS  R² = {m_ols.r2:.3f}   Moran I residuos = {mi_ols.I:+.4f} (p = {mi_ols.p_sim:.4f})")
print(f"SLX  R² = {m_slx.r2:.3f}   Moran I residuos = {mi_slx.I:+.4f} (p = {mi_slx.p_sim:.4f})")

OLS  R² = 0.669   Moran I residuos = +0.1411 (p = 0.0010)
SLX  R² = 0.687   Moran I residuos = +0.1048 (p = 0.0010)


Como vimos en clase 08, ambos modelos dejan Moran's I residual positivo y significativo. El SLX ayuda pero no resuelve la dependencia espacial. Hay que incorporarla de otra manera.

## 2. ¿SEM o SAR? Tests de Multiplicador de Lagrange (LM)

Antes de ajustar SEM o SAR a ciegas, conviene preguntar **cuál de los dos tiene más sentido para nuestros datos**. La econometría espacial responde con los **tests de multiplicador de Lagrange (LM)**.

### 2.1 ¿Qué es un test LM, en una frase?

Un test LM toma los **residuos del OLS** y pregunta:

> *"Si al modelo le faltara un término concreto (por ejemplo, $\lambda W u$ o $\rho W y$), ¿lo notaríamos en los residuos?"*

Si el p-valor del test es chico, la respuesta es **sí**: los residuos delatan que falta ese término. Lo importante: el test **no requiere ajustar SEM ni SAR** todavía. Se calcula gratis a partir del OLS, y por eso es la primera parada del diagnóstico.

El nombre viene de la técnica estadística que usa (multiplicadores de Lagrange para evaluar restricciones), pero para esta clase basta la intuición: **es un detector temprano de "algo que falta"**.

### 2.2 Los 4 tests

Hay dos cosas distintas que podrían faltar (autocorrelación en el **error** o en **y**), y de cada una hay una versión **clásica** y una **robusta**:

| Test | Pregunta concreta | Si rechaza H₀… |
|---|---|---|
| **LM-error** | ¿Falta autocorrelación en el error ($\lambda W u$)? | apunta a SEM |
| **LM-lag** | ¿Falta un lag de $y$ ($\rho W y$)? | apunta a SAR |
| **Robust LM-error** | Lo mismo que LM-error, **descontando** un posible SAR | SEM "limpio" |
| **Robust LM-lag** | Lo mismo que LM-lag, **descontando** un posible SEM | SAR "limpio" |

En la práctica los dos LM clásicos casi siempre rechazan H₀ a la vez (cualquier indicio espacial gatilla ambos). Por eso lo que decide son los **robustos**: cada uno aísla la señal de un tipo de dependencia eliminando la confusión con la otra.

### 2.3 Regla de Anselin (la estándar en la literatura)

1. Si solo uno de los LM clásicos es significativo → ese modelo.
2. Si los dos LM clásicos lo son → mirar los **robustos**: el más significativo gana.
3. Si los dos robustos también son significativos → considerar SARMA (lag + error juntos, fuera del alcance).

Veamos qué dicen nuestros datos:

In [4]:
# Re-ajustamos el OLS pidiéndole los diagnósticos espaciales (spat_diag=True).
# Es la misma regresión de antes; el flag solo le pide a spreg que calcule
# los 4 tests LM y los guarde como atributos del modelo (lm_error, lm_lag, etc.).
m_ols_diag = spreg.OLS(y, X, w=w, spat_diag=True,
                       name_y="log_price", name_x=variables)

def fmt(test):
    stat, p = test
    return f"stat = {stat:8.2f}   p = {p:.2e}"

print("Tests LM espaciales (sobre residuos de OLS):")
print(f"  LM-error          {fmt(m_ols_diag.lm_error)}")
print(f"  LM-lag            {fmt(m_ols_diag.lm_lag)}")
print(f"  Robust LM-error   {fmt(m_ols_diag.rlm_error)}")
print(f"  Robust LM-lag     {fmt(m_ols_diag.rlm_lag)}")

Tests LM espaciales (sobre residuos de OLS):
  LM-error          stat =  1350.11   p = 1.46e-295
  LM-lag            stat =  1171.38   p = 1.01e-256
  Robust LM-error   stat =   572.73   p = 1.43e-126
  Robust LM-lag     stat =   394.00   p = 1.12e-87


Los cuatro tests rechazan H₀ con p prácticamente cero. En particular, **los dos robustos también son significativos** ($p \approx 10^{-126}$ y $p \approx 10^{-87}$). Según la regla de Anselin, este es el caso ambiguo "ambos significativos" que apuntaría a **SARMA** (lag + error juntos). Como ese modelo queda fuera del alcance de la clase, vamos a hacer lo que se hace en la práctica: **ajustar SEM y SAR por separado y comparar empíricamente** (qué tanto baja Moran's I de los residuos, pseudo-R², etc.).

> **Aviso de interpretación**: a veces se dice "el LM con stat más grande gana", pero esa lectura es engañosa. Los 4 tests responden hipótesis distintas; comparar sus estadísticos como si fueran una métrica de "qué modelo es mejor" no es lo que la teoría avala. Los LM son **un detector temprano**, no la decisión final.

## 3. SEM: Modelo de Error Espacial

> $$y = X\beta + u, \quad u = \color{red}{\lambda W u} + \varepsilon$$
>
> **Qué cambia respecto al OLS**: el error $u$ ya no es ruido blanco, tiene su propio término espacial $\lambda W u$.
> **Qué buscar en el output**: el parámetro $\hat\lambda$ y su p-valor.

### 3.1 La idea

¿De dónde vienen los residuos espacialmente correlacionados? Una explicación natural: **hay variables que no medimos** y que **sí están distribuidas espacialmente**. En Airbnb, pensemos en:

- Calidad del barrio (limpieza, seguridad).
- Vista al mar o al parque.
- Distancia al transporte, restaurantes, bares.
- Ruido, tráfico.

Ninguna está en nuestra tabla, pero todas tienen una clara estructura territorial. Su efecto cae en el residuo, y al estar correlacionadas en el espacio, generan **autocorrelación espacial del error**.

### 3.2 Especificación

El modelo SEM postula que el error se descompone en una parte **espacialmente correlacionada** y una parte iid:

$$
y \;=\; X\beta \;+\; u \qquad\text{con}\qquad u \;=\; \lambda\, W u \;+\; \varepsilon
$$

- $\lambda$ es el **parámetro de autocorrelación del error**. Mide cuánto del error de un punto se explica por el error promedio de sus vecinos.
- $\varepsilon$ es ruido blanco (iid).

Si $\lambda = 0$, recuperamos OLS. Si $\lambda > 0$, los errores se contagian entre vecinos.

### 3.3 Por qué OLS no sirve

OLS sigue dando coeficientes **insesgados**, pero sus errores estándar están mal calculados porque el supuesto de independencia de los errores **se rompe**. Necesitamos un método que estime $\beta$ y $\lambda$ juntos, reconociendo la estructura de $u$.

Usamos `spreg.GM_Error_Het`: estimación por **método generalizado de los momentos (GMM)** con corrección por heteroscedasticidad. Es el estándar moderno cuando no queremos asumir homoscedasticidad (que en datos espaciales casi nunca se cumple).

### 3.4 Interpretación de $\lambda$

- $\lambda$ no afecta los $\beta$ directamente: los coeficientes de las $X$ se siguen interpretando como **efectos marginales** (cambios porcentuales en `log_price`).
- $\lambda$ resume **cuánto residuo queda por explicar** a partir del vecindario, atribuible a variables omitidas.
- $\lambda$ alto significa que el modelo se está perdiendo mucha información territorial no observada.

In [5]:
m_sem = spreg.GM_Error_Het(y, X, w=w, name_y="log_price", name_x=variables)
print(m_sem.summary)

GM_Error_Het
REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: GM SPATIALLY WEIGHTED LEAST SQUARES (HET)
------------------------------------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :   log_price                Number of Observations:        6110
Mean dependent var  :      4.9958                Number of Variables   :          12
S.D. dependent var  :      0.8072                Degrees of Freedom    :        6098
Pseudo R-squared    :      0.6655
N. of iterations    :           1                Step1c computed       :          No

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
------------------------------------------------------------------------------------
            CONSTANT         4.41524         0.03136       140.78898         0.00000
        accommodate

Lecturas clave del output:

- **`lambda`** (última fila): es nuestro $\hat\lambda$. Si es positivo y significativo, confirmamos que hay dependencia espacial en el error.
- Los coeficientes de las $X$ son muy parecidos a los del OLS (porque OLS no sesga $\beta$, solo subestima los errores estándar).
- Los **p-valores** ahora sí son confiables. Esperamos ver algunas variables que en el OLS aparecían como significativas y que ahora ya no lo son.

Verifiquemos los residuos:

**Cuidado con los residuos del SEM**: el modelo descompone $u = \lambda W u + \varepsilon$. Lo que está pensado para ser ruido blanco es $\varepsilon$ (el **residuo filtrado**), no $u$ (la diferencia entre $y$ y $X\hat\beta$). `spreg` los guarda en dos atributos distintos:

- `m_sem.u`: residuo "ingenuo" $\hat u = y - X\hat\beta$. Por construcción, sigue teniendo estructura espacial (es lo que el SEM está modelando, justamente).
- `m_sem.e_filtered`: residuo **filtrado** $\hat\varepsilon = \hat u - \hat\lambda W \hat u$. **Este** es el que debería verse como ruido blanco si el modelo es correcto.

Comparemos los dos:

In [6]:
db["residuo_sem_crudo"]    = m_sem.u.flatten()
db["residuo_sem_filtrado"] = m_sem.e_filtered.flatten()

mi_sem_u   = Moran(db["residuo_sem_crudo"].values,    w)
mi_sem_eps = Moran(db["residuo_sem_filtrado"].values, w)

print(f"Moran I residuos OLS                  = {mi_ols.I:+.4f}  (p = {mi_ols.p_sim:.4f})")
print(f"Moran I residuos SLX                  = {mi_slx.I:+.4f}  (p = {mi_slx.p_sim:.4f})")
print(f"Moran I residuos SEM crudos    (u)    = {mi_sem_u.I:+.4f}  (p = {mi_sem_u.p_sim:.4f})")
print(f"Moran I residuos SEM filtrados (eps)  = {mi_sem_eps.I:+.4f}  (p = {mi_sem_eps.p_sim:.4f})  (este es el que debe ser ~0)")

Moran I residuos OLS                  = +0.1411  (p = 0.0010)
Moran I residuos SLX                  = +0.1048  (p = 0.0010)
Moran I residuos SEM crudos    (u)    = +0.1803  (p = 0.0010)
Moran I residuos SEM filtrados (eps)  = -0.0101  (p = 0.0030)  (este es el que debe ser ~0)


Los residuos **filtrados** del SEM tienen Moran's I prácticamente cero: el modelo logró sacar la dependencia espacial de los errores. Los residuos crudos $\hat u$ siguen estructurados, y eso es esperable: el SEM no intenta eliminarlos, sino atribuirlos al término $\lambda W u$.

> Esta es una sutileza que se les escapa a muchos al leer outputs de SEM. Si solo miran `m.u`, van a concluir erróneamente que SEM no funcionó.

## 4. SAR: Modelo de Lag Espacial (con efectos directos e indirectos)

> $$y = \color{red}{\rho W y} + X\beta + \varepsilon$$
>
> **Qué cambia respecto al OLS**: el propio $y$ aparece en el lado derecho como $W y$ (el promedio de $y$ entre vecinos).
> **Qué buscar en el output**: el parámetro $\hat\rho$. Y atención: los $\beta$ del output **no son** los efectos marginales finales, hay que descomponerlos en directos e indirectos (sección 4.4).

### 4.1 La idea

SEM atribuía la dependencia espacial a **variables omitidas**. SAR propone algo distinto y más fuerte: que **el precio de una propiedad depende del precio de sus vecinas**, manteniendo todo lo demás constante.

Esto tiene sentido sustantivo en muchos contextos:

- Mercados inmobiliarios: los precios se anclan a "comparables" en la cuadra.
- Difusión social: el voto, la adopción de tecnologías, las protestas se contagian entre vecinos.
- Externalidades: un restaurante exitoso eleva los precios de las propiedades de al lado.

### 4.2 Especificación

$$
y \;=\; \rho\, W y \;+\; X\beta \;+\; \varepsilon
$$

- $\rho$ es el **parámetro de lag espacial**, también llamado *spatial autoregressive coefficient*. Mide cuánto influye el promedio de $y$ entre vecinos sobre $y_i$.

### 4.3 Por qué OLS falla aquí (y bastante)

A diferencia de SEM (donde OLS solo daba malos *standard errors*), en SAR el OLS daría coeficientes **directamente sesgados**. El problema es que $Wy$ aparece como variable explicativa, pero $Wy$ está **correlacionada con $\varepsilon$** (es un pedazo de $y$, que contiene $\varepsilon$). Eso viola el supuesto de exogeneidad, así que necesitamos **variables instrumentales**.

`spreg.GM_Lag` usa **2SLS (Two-Stage Least Squares)** con los lags de las $X$ como instrumentos de $Wy$. La idea: los atributos de los vecinos son un buen *proxy* del precio de los vecinos, pero (asumiendo exogeneidad de $X$) no están correlacionados con $\varepsilon$.

In [7]:
m_sar = spreg.GM_Lag(y, X, w=w, name_y="log_price", name_x=variables)
print(m_sar.summary)

GM_Lag
REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: SPATIAL TWO STAGE LEAST SQUARES
------------------------------------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :   log_price                Number of Observations:        6110
Mean dependent var  :      4.9958                Number of Variables   :          13
S.D. dependent var  :      0.8072                Degrees of Freedom    :        6097
Pseudo R-squared    :      0.7065
Spatial Pseudo R-squared:  0.6887

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
------------------------------------------------------------------------------------
            CONSTANT         2.70588         0.07345        36.84216         0.00000
        accommodates         0.06894         0.00482        14.31630         0.00000
 

In [8]:
db["residuo_sar"] = m_sar.u.flatten()
mi_sar = Moran(db["residuo_sar"].values, w)
print(f"Moran I residuos OLS              = {mi_ols.I:+.4f}")
print(f"Moran I residuos SLX              = {mi_slx.I:+.4f}")
print(f"Moran I residuos SEM (filtrados)  = {mi_sem_eps.I:+.4f}")
print(f"Moran I residuos SAR              = {mi_sar.I:+.4f}")

Moran I residuos OLS              = +0.1411
Moran I residuos SLX              = +0.1048
Moran I residuos SEM (filtrados)  = -0.0101
Moran I residuos SAR              = +0.0255


Tanto SEM (con residuos filtrados) como SAR logran dejar Moran's I prácticamente en cero. Cada uno por una razón distinta: SEM porque atribuye la dependencia al error y la filtra, SAR porque la atribuye al lag de $y$ y la incluye directamente como regresor.

### 4.4 Efectos directos vs indirectos: lo único que SLX no podía hacer

Aquí viene **la parte interesante del SAR**, y la razón por la que vale la pena pagar el costo de estimarlo con 2SLS.

En un OLS o un SLX, el coeficiente $\beta_k$ se lee como "cuánto cambia $y_i$ si $x_{ki}$ sube en una unidad". Punto. No hay propagación.

En SAR, en cambio, un cambio en $x_{ki}$ afecta a $y_i$ **directamente**, pero como $y_j$ depende de $y_i$ (vía $\rho W$), también afecta a los vecinos. Y esos vecinos afectan a sus vecinos, en cascada. La solución cerrada se obtiene reescribiendo el modelo:

$$
y \;=\; (I - \rho W)^{-1}\, X\beta \;+\; (I - \rho W)^{-1}\, \varepsilon
$$

El efecto marginal de $x_k$ sobre $y$ ya **no es** $\beta_k$, sino la matriz $(I - \rho W)^{-1}\beta_k$. De ahí salen tres números promedio por variable:

- **Efecto directo** (ADI): cuánto cambia $y_i$ si $x_{ki}$ sube en 1 (incluyendo el feedback que vuelve por los vecinos).
- **Efecto indirecto** (AII): cuánto cambia $y$ en los **otros** puntos por ese mismo cambio en $x_{ki}$. Es el **spillover**.
- **Efecto total** (ATI = ADI + AII): suma de ambos.

`spreg.spmultiplier` los calcula. Con el método `"simple"`, $\text{ATI} = 1/(1-\rho)$.

In [9]:
rho_hat = m_sar.betas[-1, 0]
mult = spreg.spmultiplier(w, rho_hat, method="full")

print(f"rho estimado: {rho_hat:.4f}")
print(f"  Multiplicador total    (ATI) = {mult['ati']:.4f}")
print(f"  Efecto directo medio   (ADI) = {mult['adi']:.4f}")
print(f"  Efecto indirecto medio (AII) = {mult['aii']:.4f}")

rho estimado: 0.3531
  Multiplicador total    (ATI) = 1.5457
  Efecto directo medio   (ADI) = 1.0067
  Efecto indirecto medio (AII) = 0.5390


**Cómo leer eso**: para cualquier variable explicativa $x_k$ con coeficiente estimado $\hat\beta_k$, los efectos promedio son:

| | Fórmula | Significado |
|---|---|---|
| Directo | $\hat\beta_k \cdot \text{ADI}$ | Lo que sube $y_i$ cuando $x_{ki}$ sube en 1 |
| Indirecto | $\hat\beta_k \cdot \text{AII}$ | Lo que suben los $y_j$ de los vecinos por ese cambio |
| Total | $\hat\beta_k \cdot \text{ATI}$ | Suma de ambos |

Veamos los efectos descompuestos para algunas variables clave:

In [10]:
# Tomamos los betas de SAR (sin la constante y sin rho al final)
nombres_x = m_sar.name_x[1:]   # sacamos CONSTANT
betas_x = m_sar.betas[1:-1, 0]  # sacamos constante y rho

efectos = pd.DataFrame({
    "beta_SAR":      betas_x,
    "Directo (ADI)": betas_x * mult["adi"],
    "Indirecto (AII)": betas_x * mult["aii"],
    "Total (ATI)":   betas_x * mult["ati"],
}, index=nombres_x).round(4)
efectos

,beta_SAR,Directo (ADI),Indirecto (AII),Total (ATI)
accommodates,0.0689,0.0694,0.0372,0.1066
bathrooms,0.1647,0.1658,0.0888,0.2546
bedrooms,0.1641,0.1652,0.0884,0.2536
beds,-0.0368,-0.0371,-0.0198,-0.0569
d2balboa,-0.0031,-0.0032,-0.0017,-0.0049
rt_Private_room,-0.4920,-0.4953,-0.2652,-0.7605
rt_Shared_room,-1.1170,-1.1245,-0.6021,-1.7265
pg_Condominium,0.1122,0.1129,0.0605,0.1734
pg_House,0.0053,0.0054,0.0029,0.0082
pg_Other,0.1200,0.1208,0.0647,0.1855


### 4.5 Cómo reportar el resultado: frase modelo

Esta es probablemente la subsección más práctica de toda la clase: **cómo se redacta una interpretación correcta** de un SAR. Tomemos `bedrooms` como ejemplo concreto.

#### Aviso previo

> El coeficiente Coef. SAR = +0.164 **no es el efecto real**. Es sólo el shock inicial, antes de que se propague por la red. En SAR nunca se reporta el coeficiente solo, porque subestima sistemáticamente el efecto verdadero.

#### Frase modelo

> *"En el modelo SAR, agregar un dormitorio a una propiedad de Airbnb en San Diego se asocia con un aumento de aproximadamente **+16.5% en el precio de la propia propiedad** (efecto directo, ADI). Como los precios se contagian entre vecinos ($\hat\rho = 0.353$), ese cambio también eleva en promedio **el precio de las propiedades vecinas en ~8.8%** (spillover, AII). El **efecto agregado en toda la zona** es de aproximadamente **+25.4%** (efecto total, ATI = ADI + AII)."*

#### Por qué $\hat\beta$ y ADI son casi iguales (pero no idénticos)

| | Valor | |
|---|---|---|
| $\hat\beta_{bedrooms}$ | 0.1641 | shock inicial puro |
| ADI | 0.1652 | shock + feedback que vuelve por los vecinos |

ADI = $\hat\beta \times A_{\text{diag promedio}}$ = $0.164 \times 1.007$. Ese factor 1.007 captura cuánto del shock que sale a los vecinos **regresa a uno mismo** vía la cadena de propagación. Con $\hat\rho$ moderado (0.353), el feedback es chico. Si $\hat\rho$ fuera grande (cercano a 1), ADI podría ser sustancialmente mayor que $\hat\beta$.

#### Regla práctica al reportar

1. **Nunca** reportar $\hat\beta$ solo en un SAR — no es interpretable como en OLS.
2. **Siempre** reportar $\hat\rho$ + ADI + AII (o ATI si solo hay un número).
3. Mencionar que ADI/AII/ATI son **promedios** sobre las $n$ propiedades. La librería `spreg` no devuelve efectos por zona; para eso hay que calcular $A = (I - \rho W)^{-1}$ a mano.
4. Si la variable es una dummy (`rt_Private_room`, `pg_House`, etc.), el efecto se lee comparado contra la categoría base, no como cambio marginal.

Eso es lo que SLX *no* podía darnos: un SLX captura spillovers exógenos (los atributos de los vecinos importan), pero no modela la cadena de propagación de $y$ a través de la red espacial. SAR sí.

## 5. Comparación final y cierre

Resumamos los cuatro modelos en una tabla:

In [11]:
resumen = pd.DataFrame({
    "Modelo":            ["OLS", "SLX", "SEM", "SAR"],
    "Param. espacial":   ["—", "γ (lag de X)", f"λ = {m_sem.betas[-1,0]:.3f}",
                          f"ρ = {m_sar.betas[-1,0]:.3f}"],
    "Pseudo-R²":         [round(m_ols.r2, 3), round(m_slx.r2, 3),
                          round(m_sem.pr2, 3), round(m_sar.pr2, 3)],
    "Moran I residuos":  [round(mi_ols.I, 4), round(mi_slx.I, 4),
                          round(mi_sem_eps.I, 4), round(mi_sar.I, 4)],
    "p-valor Moran":     [mi_ols.p_sim, mi_slx.p_sim, mi_sem_eps.p_sim,
                          mi_sar.p_sim],
})
resumen

,Modelo,Param. espacial,Pseudo-R²,Moran I residuos,p-valor Moran
0,OLS,—,0.669,0.1411,0.001
1,SLX,γ (lag de X),0.687,0.1048,0.001
2,SEM,λ = 0.644,0.666,-0.0101,0.003
3,SAR,ρ = 0.353,0.707,0.0255,0.001


### Cómo elegir entre los modelos

No hay una respuesta única, depende de qué pregunta intentamos responder:

| Si te interesa… | Usa… | Por qué |
|---|---|---|
| **Coeficientes interpretables**, controlando por que el espacio no rompa los errores estándar | **SEM** | Limpia inferencia, $\beta$ se leen como siempre |
| **Modelar contagio** (un precio influye al de al lado), efectos directos e indirectos | **SAR** | Es el único que descompone propagación |
| **Spillovers atribuibles a atributos del entorno** (vecinos grandes vs pequeños) | **SLX** | Lo más simple e interpretable, OLS estándar |
| Diagnóstico inicial | **Tests LM** | Te dice si SEM, SAR, o ninguno |

### Lo que viene en la clase 10

Una quinta forma de incorporar la dependencia espacial que dejamos para la próxima clase: **regímenes / heterogeneidad espacial**. La idea es dejar que los $\beta$ mismos cambien por zona (por ejemplo, costa vs interior), y testear con un **Chow test** si la diferencia es significativa. La intuición: quizá una habitación extra suma más al precio en La Jolla que en El Cajon, y conviene modelar eso explícitamente.

### Ideas que quedan abiertas (no las usaremos esta clase)

- **SARMA** (Spatial AutoRegressive Moving Average): combina **SAR + SEM** en una sola ecuación, $\;y = \rho W y + X\beta + u\;$ con $\;u = \lambda W u + \varepsilon$. Es el modelo natural cuando los dos LM robustos rechazan a la vez (justo nuestro caso). En `spreg` se llama `GM_Combo_Het`. Vale la pena tenerlo en mente para proyectos donde ambos tipos de dependencia conviven.
- **SDM** (Spatial Durbin Model): combina **SAR + SLX**, $\;y = \rho W y + X\beta + WX\gamma + \varepsilon$. Parte de la literatura econométrica reciente lo recomienda como modelo "por defecto".
- **ML estimation**: alternativa a GMM (`spreg.ML_Lag`, `spreg.ML_Error`). Más eficiente con datos pequeños y errores normales.
- **GWR** (Geographically Weighted Regression): la versión extrema de regímenes, un set distinto de $\beta$ por **cada** punto. Lo veremos en la clase 16.

## Para profundizar

- Rey, S., Arribas-Bel, D. & Wolf, L. J. (2020). *Geographic Data Science with Python*, Capítulo 11.
- Anselin, L. (1988). *Spatial Econometrics: Methods and Models*. Kluwer Academic Publishers.
- LeSage, J. & Pace, R. K. (2009). *Introduction to Spatial Econometrics*. CRC Press.
- Elhorst, J. P. (2014). *Spatial Econometrics: From Cross-Sectional Data to Spatial Panels*. Springer.